# Original Code (Before Optimization)

In [ ]:
from __future__ import print_function

import numpy as np


def cnd(d):
    A1 = 0.31938153
    A2 = -0.356563782
    A3 = 1.781477937
    A4 = -1.821255978
    A5 = 1.330274429
    RSQRT2PI = 0.39894228040143267793994605993438
    K = 1.0 / (1.0 + 0.2316419 * np.abs(d))
    ret_val = (RSQRT2PI * np.exp(-0.5 * d * d) *
               (K * (A1 + K * (A2 + K * (A3 + K * (A4 + K * A5))))))
    return np.where(d > 0, 1.0 - ret_val, ret_val)

    # SPEEDTIP: Despite the memory overhead and redundant computation, the above
    # is much faster than:
    #
    # for i in range(len(d)):
    #     if d[i] > 0:
    #         ret_val[i] = 1.0 - ret_val[i]
    # return ret_val


def black_scholes(stockPrice, optionStrike, optionYears, Riskfree, Volatility):
    S = stockPrice
    X = optionStrike
    T = optionYears
    R = Riskfree
    V = Volatility
    sqrtT = np.sqrt(T)
    d1 = (np.log(S / X) + (R + 0.5 * V * V) * T) / (V * sqrtT)
    d2 = d1 - V * sqrtT
    cndd1 = cnd(d1)
    cndd2 = cnd(d2)

    expRT = np.exp(- R * T)

    callResult = S * cndd1 - X * expRT * cndd2
    putResult = X * expRT * (1.0 - cndd2) - S * (1.0 - cndd1)

    return callResult, putResult


# OPTIMISED CODE 
## Using CUDA STREAMS + Pinned Memory

In [1]:
import numpy as np
import time
from numba import cuda
import math

# --- CPU Version (unchanged) ---
def cnd_cpu(d):
    A1 = 0.31938153; A2 = -0.356563782; A3 = 1.781477937
    A4 = -1.821255978; A5 = 1.330274429
    RSQRT2PI = 0.39894228040143267793994605993438
    K = 1.0 / (1.0 + 0.2316419 * np.abs(d))
    ret_val = (RSQRT2PI * np.exp(-0.5 * d * d) *
               (K * (A1 + K * (A2 + K * (A3 + K * (A4 + K * A5))))))
    return np.where(d > 0, 1.0 - ret_val, ret_val)

def black_scholes_cpu(stockPrice, optionStrike, optionYears, Riskfree, Volatility):
    S, X, T, R, V = stockPrice, optionStrike, optionYears, Riskfree, Volatility
    sqrtT = np.sqrt(T)
    d1 = (np.log(S / X) + (R + 0.5 * V * V) * T) / (V * sqrtT)
    d2 = d1 - V * sqrtT
    cndd1 = cnd_cpu(d1); cndd2 = cnd_cpu(d2)
    expRT = np.exp(-R * T)
    return S * cndd1 - X * expRT * cndd2, X * expRT * (1.0 - cndd2) - S * (1.0 - cndd1)

# --- GPU Kernel (unchanged) ---
@cuda.jit(device=True, inline=True, fastmath=True)
def cnd_device(d):
    A1 = 0.31938153; A2 = -0.356563782; A3 = 1.781477937
    A4 = -1.821255978; A5 = 1.330274429
    RSQRT2PI = 0.39894228040143267793994605993438
    K = 1.0 / (1.0 + 0.2316419 * math.fabs(d))
    ret_val = (RSQRT2PI * math.exp(-0.5 * d * d) *
               (K * (A1 + K * (A2 + K * (A3 + K * (A4 + K * A5))))))
    return (1.0 - ret_val) if d > 0 else ret_val

@cuda.jit(fastmath=True)
def black_scholes_kernel(stockPrice, optionStrike, optionYears, Riskfree, Volatility,
                          callResult, putResult):
    tid = cuda.grid(1)
    N = stockPrice.shape[0]
    if tid < N:
        S, X, T, R, V = stockPrice[tid], optionStrike[tid], optionYears[tid], Riskfree[tid], Volatility[tid]
        sqrtT = math.sqrt(T)
        d1 = (math.log(S / X) + (R + 0.5 * V * V) * T) / (V * sqrtT)
        d2 = d1 - V * sqrtT
        cndd1 = cnd_device(d1); cndd2 = cnd_device(d2)
        expRT = math.exp(-R * T)
        callResult[tid] = S * cndd1 - X * expRT * cndd2
        putResult[tid]  = X * expRT * (1.0 - cndd2) - S * (1.0 - cndd1)


def run_benchmark(num_options=10**7, iterations=5, num_streams=4):

    np.random.seed(42)
    stockPrice   = np.random.uniform(50,   150,  num_options).astype(np.float64)
    optionStrike = np.random.uniform(50,   150,  num_options).astype(np.float64)
    optionYears  = np.random.uniform(0.5,  2.0,  num_options).astype(np.float64)
    Riskfree     = np.random.uniform(0.01, 0.05, num_options).astype(np.float64)
    Volatility   = np.random.uniform(0.1,  0.5,  num_options).astype(np.float64)

    print(f"Benchmarking with {num_options} options, {iterations} iterations, {num_streams} streams.")

    # ── CPU Benchmark ─────────────────────────────────────────────────────────
    cpu_call_result, cpu_put_result = black_scholes_cpu(
        stockPrice, optionStrike, optionYears, Riskfree, Volatility)  # warm-up

    cpu_times = []
    print("\n--- Starting CPU Benchmark ---")
    for i in range(iterations):
        t0 = time.perf_counter()
        cpu_call_result, cpu_put_result = black_scholes_cpu(
            stockPrice, optionStrike, optionYears, Riskfree, Volatility)
        cpu_times.append(time.perf_counter() - t0)
        print(f"  CPU iteration {i+1}: {cpu_times[-1]:.4f}s")
    avg_cpu_time = np.mean(cpu_times)
    print(f"Average CPU time: {avg_cpu_time:.4f}s")

    if not cuda.is_available():
        print("\nCUDA not available. Skipping GPU benchmark.")
        return

    # ── Pinned Memory Allocation ──────────────────────────────────────────────
    # Input arrays: pinned pageable→pinned copy
    pin_stockPrice   = cuda.pinned_array(num_options, dtype=np.float64)
    pin_optionStrike = cuda.pinned_array(num_options, dtype=np.float64)
    pin_optionYears  = cuda.pinned_array(num_options, dtype=np.float64)
    pin_Riskfree     = cuda.pinned_array(num_options, dtype=np.float64)
    pin_Volatility   = cuda.pinned_array(num_options, dtype=np.float64)
    pin_stockPrice[:]   = stockPrice
    pin_optionStrike[:] = optionStrike
    pin_optionYears[:]  = optionYears
    pin_Riskfree[:]     = Riskfree
    pin_Volatility[:]   = Volatility

    # Output arrays: pinned so D2H is also fast
    gpu_call_result = cuda.pinned_array(num_options, dtype=np.float64)
    gpu_put_result  = cuda.pinned_array(num_options, dtype=np.float64)

    # ── Stream Setup ──────────────────────────────────────────────────────────
    # Divide work evenly; last chunk absorbs any remainder
    chunk_size = (num_options + num_streams - 1) // num_streams

    streams = [cuda.stream() for _ in range(num_streams)]

    # Pre-allocate one set of GPU arrays per stream (reused every iteration)
    # This avoids repeated GPU malloc inside the timed loop
    d_in  = []   # list of [d_S, d_X, d_T, d_R, d_V] per stream
    d_out = []   # list of [d_call, d_put]           per stream

    for s in range(num_streams):
        start = s * chunk_size
        end   = min(start + chunk_size, num_options)
        size  = end - start
        d_in.append([
            cuda.device_array(size, dtype=np.float64),  # stockPrice
            cuda.device_array(size, dtype=np.float64),  # optionStrike
            cuda.device_array(size, dtype=np.float64),  # optionYears
            cuda.device_array(size, dtype=np.float64),  # Riskfree
            cuda.device_array(size, dtype=np.float64),  # Volatility
        ])
        d_out.append([
            cuda.device_array(size, dtype=np.float64),  # callResult
            cuda.device_array(size, dtype=np.float64),  # putResult
        ])

    tpb = 256  # threads per block

    # Helper: dispatch one full pipeline pass across all streams
    def dispatch_all_streams():
        for s in range(num_streams):
            start  = s * chunk_size
            end    = min(start + chunk_size, num_options)
            size   = end - start
            blocks = (size + tpb - 1) // tpb
            st     = streams[s]
            si, so = d_in[s], d_out[s]

            # ① Async H2D: pinned host chunk → GPU (no CPU stall)
            si[0].copy_to_device(pin_stockPrice[start:end],   stream=st)
            si[1].copy_to_device(pin_optionStrike[start:end], stream=st)
            si[2].copy_to_device(pin_optionYears[start:end],  stream=st)
            si[3].copy_to_device(pin_Riskfree[start:end],     stream=st)
            si[4].copy_to_device(pin_Volatility[start:end],   stream=st)

            # ② Async Kernel: runs after H2D completes ON THIS STREAM
            black_scholes_kernel[blocks, tpb, st](
                si[0], si[1], si[2], si[3], si[4], so[0], so[1])

            # ③ Async D2H: runs after kernel completes ON THIS STREAM
            so[0].copy_to_host(gpu_call_result[start:end], stream=st)
            so[1].copy_to_host(gpu_put_result[start:end],  stream=st)
        # All 3 stages queued for all streams; GPU overlaps them automatically

    # ── Warm-Up (3 passes, fully synchronised) ───────────────────────────────
    print("\n--- GPU Streams Warm Up Started ---")
    for _ in range(3):
        dispatch_all_streams()
        cuda.synchronize()
    print("--- GPU Streams Warm Up Done ---")

    # ── Streamed Benchmark ────────────────────────────────────────────────────
    stream_times = []
    print("\n--- Starting GPU Streams Benchmark ---")
    for i in range(iterations):
        t0 = time.perf_counter()
        dispatch_all_streams()
        cuda.synchronize()          # wait for EVERY stream to finish
        stream_times.append(time.perf_counter() - t0)
        print(f"  Streams iteration {i+1}: {stream_times[-1]:.4f}s")

    avg_stream_time = np.mean(stream_times)
    print(f"Average Streams time (H2D+Kernel+D2H): {avg_stream_time:.4f}s")

    # ── Results & Speedups ────────────────────────────────────────────────────
    stream_speedup = avg_cpu_time / avg_stream_time
    print(f"\n--- Performance Summary ---")
    print(f"GPU Streams is {stream_speedup:.2f}x faster than CPU (full pipeline).")

    # ── Verification ──────────────────────────────────────────────────────────
    print("\n--- Verifying Results ---")
    sample_indices = np.random.choice(num_options, 10, replace=False)
    print("Sample Call — CPU:", cpu_call_result[sample_indices])
    print("Sample Call — GPU:", gpu_call_result[sample_indices])

    call_ok = np.allclose(cpu_call_result, gpu_call_result, rtol=1e-4, atol=1e-6)
    put_ok  = np.allclose(cpu_put_result,  gpu_put_result,  rtol=1e-4, atol=1e-6)
    if call_ok and put_ok:
        print("Results match between CPU and GPU (within tolerance).")
    else:
        print("WARNING: Results do NOT match.")
        print(f"  Max call diff: {np.max(np.abs(cpu_call_result - gpu_call_result)):.2e}")
        print(f"  Max put  diff: {np.max(np.abs(cpu_put_result  - gpu_put_result )):.2e}")


run_benchmark(num_options=(10**8) * 1, iterations=5, num_streams=8)


Benchmarking with 100000000 options, 5 iterations, 8 streams.

--- Starting CPU Benchmark ---
  CPU iteration 1: 12.8255s
  CPU iteration 2: 12.8108s
  CPU iteration 3: 12.9675s
  CPU iteration 4: 12.8233s
  CPU iteration 5: 12.9297s
Average CPU time: 12.8714s

--- GPU Streams Warm Up Started ---
--- GPU Streams Warm Up Done ---

--- Starting GPU Streams Benchmark ---
  Streams iteration 1: 0.3747s
  Streams iteration 2: 0.3740s
  Streams iteration 3: 0.3753s
  Streams iteration 4: 0.3744s
  Streams iteration 5: 0.3744s
Average Streams time (H2D+Kernel+D2H): 0.3745s

--- Performance Summary ---
GPU Streams is 34.37x faster than CPU (full pipeline).

--- Verifying Results ---
Sample Call — CPU: [10.97212405  8.98799198 49.10857824 21.64761811 13.31766179 12.60110805
 25.82739995 29.85409807  2.31476154 43.02943625]
Sample Call — GPU: [10.97212405  8.98799198 49.10857824 21.64761811 13.31766179 12.60110805
 25.82739995 29.85409807  2.31476154 43.02943625]
Results match between CPU and GP

In [1]:
import numpy as np
import time
from numba import cuda
import math

# --- CPU Version (unchanged) ---
def cnd_cpu(d):
    A1 = 0.31938153; A2 = -0.356563782; A3 = 1.781477937
    A4 = -1.821255978; A5 = 1.330274429
    RSQRT2PI = 0.39894228040143267793994605993438
    K = 1.0 / (1.0 + 0.2316419 * np.abs(d))
    ret_val = (RSQRT2PI * np.exp(-0.5 * d * d) *
               (K * (A1 + K * (A2 + K * (A3 + K * (A4 + K * A5))))))
    return np.where(d > 0, 1.0 - ret_val, ret_val)

def black_scholes_cpu(stockPrice, optionStrike, optionYears, Riskfree, Volatility):
    S, X, T, R, V = stockPrice, optionStrike, optionYears, Riskfree, Volatility
    sqrtT = np.sqrt(T)
    d1 = (np.log(S / X) + (R + 0.5 * V * V) * T) / (V * sqrtT)
    d2 = d1 - V * sqrtT
    cndd1 = cnd_cpu(d1); cndd2 = cnd_cpu(d2)
    expRT = np.exp(-R * T)
    return S * cndd1 - X * expRT * cndd2, X * expRT * (1.0 - cndd2) - S * (1.0 - cndd1)

# --- GPU Kernel (unchanged) ---
@cuda.jit(device=True, inline=True, fastmath=True)
def cnd_device(d):
    A1 = 0.31938153; A2 = -0.356563782; A3 = 1.781477937
    A4 = -1.821255978; A5 = 1.330274429
    RSQRT2PI = 0.39894228040143267793994605993438
    K = 1.0 / (1.0 + 0.2316419 * math.fabs(d))
    ret_val = (RSQRT2PI * math.exp(-0.5 * d * d) *
               (K * (A1 + K * (A2 + K * (A3 + K * (A4 + K * A5))))))
    return (1.0 - ret_val) if d > 0 else ret_val

@cuda.jit(fastmath=True)
def black_scholes_kernel(stockPrice, optionStrike, optionYears, Riskfree, Volatility,
                          callResult, putResult):
    tid = cuda.grid(1)
    N = stockPrice.shape[0]
    if tid < N:
        S, X, T, R, V = stockPrice[tid], optionStrike[tid], optionYears[tid], Riskfree[tid], Volatility[tid]
        sqrtT = math.sqrt(T)
        d1 = (math.log(S / X) + (R + 0.5 * V * V) * T) / (V * sqrtT)
        d2 = d1 - V * sqrtT
        cndd1 = cnd_device(d1); cndd2 = cnd_device(d2)
        expRT = math.exp(-R * T)
        callResult[tid] = S * cndd1 - X * expRT * cndd2
        putResult[tid]  = X * expRT * (1.0 - cndd2) - S * (1.0 - cndd1)


def run_benchmark(num_options=10**7, iterations=5, num_streams=4):

    np.random.seed(42)
    stockPrice   = np.random.uniform(50,   150,  num_options).astype(np.float64)
    optionStrike = np.random.uniform(50,   150,  num_options).astype(np.float64)
    optionYears  = np.random.uniform(0.5,  2.0,  num_options).astype(np.float64)
    Riskfree     = np.random.uniform(0.01, 0.05, num_options).astype(np.float64)
    Volatility   = np.random.uniform(0.1,  0.5,  num_options).astype(np.float64)

    print(f"Benchmarking with {num_options} options, {iterations} iterations, {num_streams} streams.")

    # ── CPU Benchmark ─────────────────────────────────────────────────────────
    cpu_call_result, cpu_put_result = black_scholes_cpu(
        stockPrice, optionStrike, optionYears, Riskfree, Volatility)  # warm-up

    cpu_times = []
    print("\n--- Starting CPU Benchmark ---")
    for i in range(iterations):
        t0 = time.perf_counter()
        cpu_call_result, cpu_put_result = black_scholes_cpu(
            stockPrice, optionStrike, optionYears, Riskfree, Volatility)
        cpu_times.append(time.perf_counter() - t0)
        print(f"  CPU iteration {i+1}: {cpu_times[-1]:.4f}s")
    avg_cpu_time = np.mean(cpu_times)
    print(f"Average CPU time: {avg_cpu_time:.4f}s")

    if not cuda.is_available():
        print("\nCUDA not available. Skipping GPU benchmark.")
        return

    # ── Pinned Memory Allocation ──────────────────────────────────────────────
    # Input arrays: pinned pageable→pinned copy
    pin_stockPrice   = cuda.pinned_array(num_options, dtype=np.float64)
    pin_optionStrike = cuda.pinned_array(num_options, dtype=np.float64)
    pin_optionYears  = cuda.pinned_array(num_options, dtype=np.float64)
    pin_Riskfree     = cuda.pinned_array(num_options, dtype=np.float64)
    pin_Volatility   = cuda.pinned_array(num_options, dtype=np.float64)
    pin_stockPrice[:]   = stockPrice
    pin_optionStrike[:] = optionStrike
    pin_optionYears[:]  = optionYears
    pin_Riskfree[:]     = Riskfree
    pin_Volatility[:]   = Volatility

    # Output arrays: pinned so D2H is also fast
    gpu_call_result = cuda.pinned_array(num_options, dtype=np.float64)
    gpu_put_result  = cuda.pinned_array(num_options, dtype=np.float64)

    # ── Stream Setup ──────────────────────────────────────────────────────────
    # Divide work evenly; last chunk absorbs any remainder
    chunk_size = (num_options + num_streams - 1) // num_streams

    streams = [cuda.stream() for _ in range(num_streams)]

    # Pre-allocate one set of GPU arrays per stream (reused every iteration)
    # This avoids repeated GPU malloc inside the timed loop
    d_in  = []   # list of [d_S, d_X, d_T, d_R, d_V] per stream
    d_out = []   # list of [d_call, d_put]           per stream

    for s in range(num_streams):
        start = s * chunk_size
        end   = min(start + chunk_size, num_options)
        size  = end - start
        d_in.append([
            cuda.device_array(size, dtype=np.float64),  # stockPrice
            cuda.device_array(size, dtype=np.float64),  # optionStrike
            cuda.device_array(size, dtype=np.float64),  # optionYears
            cuda.device_array(size, dtype=np.float64),  # Riskfree
            cuda.device_array(size, dtype=np.float64),  # Volatility
        ])
        d_out.append([
            cuda.device_array(size, dtype=np.float64),  # callResult
            cuda.device_array(size, dtype=np.float64),  # putResult
        ])

    tpb = 256  # threads per block

    # Helper: dispatch one full pipeline pass across all streams
    def dispatch_all_streams():
        for s in range(num_streams):
            start  = s * chunk_size
            end    = min(start + chunk_size, num_options)
            size   = end - start
            blocks = (size + tpb - 1) // tpb
            st     = streams[s]
            si, so = d_in[s], d_out[s]

            # ① Async H2D: pinned host chunk → GPU (no CPU stall)
            si[0].copy_to_device(pin_stockPrice[start:end],   stream=st)
            si[1].copy_to_device(pin_optionStrike[start:end], stream=st)
            si[2].copy_to_device(pin_optionYears[start:end],  stream=st)
            si[3].copy_to_device(pin_Riskfree[start:end],     stream=st)
            si[4].copy_to_device(pin_Volatility[start:end],   stream=st)

            # ② Async Kernel: runs after H2D completes ON THIS STREAM
            black_scholes_kernel[blocks, tpb, st](
                si[0], si[1], si[2], si[3], si[4], so[0], so[1])

            # ③ Async D2H: runs after kernel completes ON THIS STREAM
            so[0].copy_to_host(gpu_call_result[start:end], stream=st)
            so[1].copy_to_host(gpu_put_result[start:end],  stream=st)
        # All 3 stages queued for all streams; GPU overlaps them automatically

    # ── Warm-Up (3 passes, fully synchronised) ───────────────────────────────
    print("\n--- GPU Streams Warm Up Started ---")
    for _ in range(3):
        dispatch_all_streams()
        cuda.synchronize()
    print("--- GPU Streams Warm Up Done ---")

    # ── Streamed Benchmark ────────────────────────────────────────────────────
    stream_times = []
    print("\n--- Starting GPU Streams Benchmark ---")
    for i in range(iterations):
        t0 = time.perf_counter()
        dispatch_all_streams()
        cuda.synchronize()          # wait for EVERY stream to finish
        stream_times.append(time.perf_counter() - t0)
        print(f"  Streams iteration {i+1}: {stream_times[-1]:.4f}s")

    avg_stream_time = np.mean(stream_times)
    print(f"Average Streams time (H2D+Kernel+D2H): {avg_stream_time:.4f}s")

    # ── Results & Speedups ────────────────────────────────────────────────────
    stream_speedup = avg_cpu_time / avg_stream_time
    print(f"\n--- Performance Summary ---")
    print(f"GPU Streams is {stream_speedup:.2f}x faster than CPU (full pipeline).")

    # ── Verification ──────────────────────────────────────────────────────────
    print("\n--- Verifying Results ---")
    sample_indices = np.random.choice(num_options, 10, replace=False)
    print("Sample Call — CPU:", cpu_call_result[sample_indices])
    print("Sample Call — GPU:", gpu_call_result[sample_indices])

    call_ok = np.allclose(cpu_call_result, gpu_call_result, rtol=1e-4, atol=1e-6)
    put_ok  = np.allclose(cpu_put_result,  gpu_put_result,  rtol=1e-4, atol=1e-6)
    if call_ok and put_ok:
        print("Results match between CPU and GPU (within tolerance).")
    else:
        print("WARNING: Results do NOT match.")
        print(f"  Max call diff: {np.max(np.abs(cpu_call_result - gpu_call_result)):.2e}")
        print(f"  Max put  diff: {np.max(np.abs(cpu_put_result  - gpu_put_result )):.2e}")


run_benchmark(num_options=(10**8) * 2, iterations=5, num_streams=8)


Benchmarking with 200000000 options, 5 iterations, 8 streams.

--- Starting CPU Benchmark ---
  CPU iteration 1: 25.9202s
  CPU iteration 2: 25.8064s
  CPU iteration 3: 25.9018s
  CPU iteration 4: 25.7031s
  CPU iteration 5: 25.7759s
Average CPU time: 25.8215s

--- GPU Streams Warm Up Started ---
--- GPU Streams Warm Up Done ---

--- Starting GPU Streams Benchmark ---
  Streams iteration 1: 0.7488s
  Streams iteration 2: 0.7482s
  Streams iteration 3: 0.7492s
  Streams iteration 4: 0.7478s
  Streams iteration 5: 0.7491s
Average Streams time (H2D+Kernel+D2H): 0.7486s

--- Performance Summary ---
GPU Streams is 34.49x faster than CPU (full pipeline).

--- Verifying Results ---
Sample Call — CPU: [10.0332485  84.86172617  0.8517943  40.12633017  1.49828782 41.38112611
 46.07402779 32.82489325 61.25152044 18.40847933]
Sample Call — GPU: [10.0332485  84.86172617  0.8517943  40.12633017  1.49828782 41.38112611
 46.07402779 32.82489325 61.25152044 18.40847933]
Results match between CPU and GP